#**Module 21 — AI Agent for LinkedIn Post Generation**

###**Program: Ostad AI/ML Engineering Program (Batch 6)**

###**Author: Farjana Ferdausi**

**Stack : LangChain (LCEL) & Google Gemini (`gemini-3.6-flash`)**

**About this notebook :**

- Provide a `topic` and a `language`, the agent returns a publish-ready LinkedIn post (2–4 paragraphs, hook, hashtags, call to action).

- This is an `agent`, not just a prompt.

- Instead of one LLM call, the agent runs a `two-stage reflection pipeline`


1. Draft — the LLM writes a first version.

2. Critique & refine — the LLM reviews its **own** draft against a
   LinkedIn best-practices checklist and returns an improved final version.

```
topic, language ──▶ [ Draft Chain ] ──▶ [ Critique & Refine Chain ] ──▶ final post
                       (LLM call 1)            (LLM call 2)
```

`Self-correction loop is what turns a plain chain into agentic behavior.``

## 1. Install dependencies

In [1]:
!pip install -q -U langchain langchain-core langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 8.7 MB/s eta 0:00:00


##**2. Set - Gemini API key :**

- I created a **free** `API-key` from Google AI Studio and used it to connect the Gemini model with my project.


In [2]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("Loaded GOOGLE_API_KEY from Colab secrets.")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")
    print("Loaded GOOGLE_API_KEY from manual input.")

Loaded GOOGLE_API_KEY from Colab secrets.


##**3. Build the agent :**

- For this project, I created the main agent logic in a separate Python file called (`agent.py`).This is standard practice in production
ML engineering — notebooks are for demos and exploration, (`.py`) files hold the
reusable logic.

- I did this to keep the notebook simple and organized.

- The notebook is mainly used for testing and demonstrating the agent, while (`agent.py`) contains the reusable code also for generating LinkedIn posts.

In [4]:
%%writefile agent.py
"""
agent.py

An AI agent, built with LangChain + Google Gemini, that turns a
(topic, language) pair into a publish-ready LinkedIn post.

--------------------------------------------------------------------
WHY THIS COUNTS AS AN "AGENT" AND NOT JUST A SINGLE PROMPT
--------------------------------------------------------------------
A single prompt -> LLM -> output call is just a "chain." This project
instead uses a two-stage REFLECTION pattern, which is a recognized
agentic design used in real production systems:

    Stage 1 (DRAFT)    : the LLM writes a first version of the post.
    Stage 2 (CRITIQUE) : the LLM re-reads its OWN draft against a
                          checklist of LinkedIn best practices and
                          returns an improved final version.

The agent is, in effect, reasoning about and correcting its own work
before handing it back to the user - that self-correction loop is
what separates an "agent" from a plain prompt-response call.

Author: Farjana Ferdausi
Module: Ostad AI/ML Engineering Program - Module 21
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import Runnable
from langchain_google_genai import ChatGoogleGenerativeAI


# A default model. gemini-3.6-flash is on Google's free tier
# and is the current stable/GA workhorse Flash model.
# Note on model lifecycles: Google regularly retires older model IDs
# (e.g. gemini-2.5-flash was retired for new users shortly after this
# project started). If generate() ever raises a 404 / "model not found"
# error, check the current model list at https://ai.google.dev/gemini-api/docs/models
# and update this ONE line - no other code needs to change, which is the
# whole point of keeping the model name in a single constant.
DEFAULT_MODEL = "gemini-3.6-flash"


@dataclass
class LinkedInPost:
    """
    A small container that holds everything about one generated post.

    Keeping the draft AND the final version (instead of just the final
    text) makes it easy to show, in the demo video, exactly what the
    'critique' stage changed - this is good evidence that the agent is
    really doing multi-step reasoning, not just calling the API once.
    """
    topic: str
    language: str
    draft: str
    final: str


class LinkedInPostAgent:
    """
    Usage
    -----
    >>> agent = LinkedInPostAgent()
    >>> post = agent.generate(topic="AI in Healthcare", language="English")
    >>> print(post.final)
    """

    def __init__(self, model_name: str = DEFAULT_MODEL, temperature: float = 0.7):
        if not os.environ.get("GOOGLE_API_KEY"):
            raise ValueError(
                "GOOGLE_API_KEY is not set. Get a free key from "
                "https://aistudio.google.com/apikey and set it as an "
                "environment variable (see README.md) before creating the agent."
            )

        # ChatGoogleGenerativeAI reads the GOOGLE_API_KEY environment variable
        # automatically - this is intentional. Passing the key as an explicit
        # keyword argument is possible too, but the parameter name has changed
        # between langchain-google-genai versions; the env var is the one
        # interface Google and LangChain both guarantee stays stable.
        self.llm = ChatGoogleGenerativeAI(
            model=model_name,
            temperature=temperature,
        )

        # Build both stages of the pipeline once, at start-up, so that
        # generate() just re-uses them instead of rebuilding on every call.
        self.draft_chain: Runnable = self._build_draft_chain()
        self.critique_chain: Runnable = self._build_critique_chain()

    # ------------------------------------------------------------------
    # Stage 1: DRAFT
    # ------------------------------------------------------------------
    def _build_draft_chain(self) -> Runnable:
        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You are a senior LinkedIn content strategist who writes "
                    "concise, professional, engaging posts. "
                    "You ALWAYS write the post entirely in the language the "
                    "user asks for - never default to English unless English "
                    "is the language requested.",
                ),
                (
                    "human",
                    "Write a LinkedIn post about: {topic}\n"
                    "Language: {language}\n\n"
                    "Requirements:\n"
                    "- 2 to 4 short paragraphs\n"
                    "- Open with a strong hook (first line must earn a click on "
                    '"see more")\n'
                    "- Use short paragraphs and line breaks, the way real "
                    "LinkedIn posts are formatted\n"
                    "- Close with a call to action or a question that invites "
                    "comments\n"
                    "- End with 3 to 5 relevant hashtags\n"
                    "- Plain text only - no markdown symbols like ** or ##",
                ),
            ]
        )
        return prompt | self.llm | StrOutputParser()

    # ------------------------------------------------------------------
    # Stage 2: CRITIQUE & REFINE
    # ------------------------------------------------------------------
    def _build_critique_chain(self) -> Runnable:
        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You are a strict LinkedIn editor. You improve drafts "
                    "without changing their language or their core message. "
                    "Reply with ONLY the improved post - no notes, no "
                    "preamble, no explanations.",
                ),
                (
                    "human",
                    "Topic: {topic}\n"
                    "Language: {language}\n"
                    "Draft post:\n{draft}\n\n"
                    "Review the draft against this checklist, then return the "
                    "improved final version:\n"
                    "1. Is the opening line strong enough to stop someone "
                    "mid-scroll?\n"
                    "2. Is it broken into short, easy-to-scan paragraphs?\n"
                    "3. Does it sound like a real professional, not a robot?\n"
                    "4. Are the hashtags relevant and limited to 3-5?",
                ),
            ]
        )
        return prompt | self.llm | StrOutputParser()

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------
    def generate(self, topic: str, language: str = "English") -> LinkedInPost:
        """Run the full draft -> critique -> refine pipeline once."""
        if not topic or not topic.strip():
            raise ValueError("Topic cannot be empty.")
        if not language or not language.strip():
            raise ValueError("Language cannot be empty.")

        draft = self.draft_chain.invoke({"topic": topic, "language": language})
        final = self.critique_chain.invoke(
            {"topic": topic, "language": language, "draft": draft}
        )

        return LinkedInPost(topic=topic, language=language, draft=draft, final=final)


Overwriting agent.py


In [5]:
from agent import LinkedInPostAgent, LinkedInPost

agent = LinkedInPostAgent()
print("Agent ready. Model:", agent.llm.model)

Agent ready. Model: gemini-3.6-flash


##**4. Try it yourself :**

- Now I can test the agent by changing the topic and language below. No code changes are needed. I just enter my choices in the Colab form and run the cell to generate a LinkedIn post.

In [6]:
#@title Generate a LinkedIn post
topic = "AI in Healthcare"  #@param {type:"string"}
language = "English"  #@param ["English", "Bengali", "Spanish", "Hindi", "French", "Arabic"]

result = agent.generate(topic=topic, language=language)

print("=" * 60)
print(f"TOPIC: {result.topic}   |   LANGUAGE: {result.language}")
print("=" * 60)
print(result.final)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


TOPIC: AI in Healthcare   |   LANGUAGE: English
The biggest revolution in modern healthcare isn’t a breakthrough drug. 

It’s code.

From predicting patient complications hours before they happen to cutting administrative paperwork in half, AI is quietly transforming clinical workflows. 

But the real power of AI in medicine isn't about replacing human expertise. 

It's about giving doctors and nurses their most valuable asset back: time with patients.

The question is no longer whether AI belongs in hospitals. It's how fast health systems can integrate it safely, ethically, and effectively.

Where do you see AI having the most immediate impact right now—diagnostics, operations, or direct patient care?

Drop your thoughts below.

#AIinHealthcare #HealthTech #DigitalHealth #HealthcareInnovation


##**5. See the agent's reasoning: draft → refined :**

- This cell shows how the agent works in two steps. First, it creates a draft of the LinkedIn post, and then it improves and refines the draft.

- I can show both steps in the demo video to explain how the agent generates the final post.

In [7]:
demo = agent.generate(topic="The Future of Remote Work", language="English")

print("STAGE 1 — First Draft")
print("-" * 60)
print(demo.draft)

print("\n\nSTAGE 2 — After Self-Critique & Refinement")
print("-" * 60)
print(demo.final)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


STAGE 1 — First Draft
------------------------------------------------------------
The debate over remote work is asking the wrong question.

It was never just about working from home versus sitting in a cubicle. The true evolution of work is about asynchronous collaboration, output-based evaluation, and giving high performers autonomy over their time.

Companies forcing rigid return-to-office mandates risk losing their best talent to organizations that prioritize trust over physical presence. The winners of the next decade won't be the ones with the most impressive real estate—they will be the ones with the most adaptable digital infrastructure and culture.

Is your organization doubling down on flexibility, or pushing for a full return to the office?

#RemoteWork #FutureOfWork #HybridWork #WorkplaceCulture #Leadership


STAGE 2 — After Self-Critique & Refinement
------------------------------------------------------------
Most leaders are asking the wrong question about remote work.


##**6. Multi-language demo :**

- The agent can generate LinkedIn posts in different languages. In this example, I tested it with `English`, `Bengali`, and `Spanish`. The same agent is used for all `three languages`, and each post is generated only once.

In [8]:
demo_examples = [
    ("AI in Healthcare", "English"),
    ("Remote Work Productivity", "Bengali"),
    ("The Future of Renewable Energy", "Spanish"),
]

results = [agent.generate(topic=t, language=l) for t, l in demo_examples]

for r in results:
    print("\n" + "=" * 60)
    print(f"{r.language.upper()} — {r.topic}")
    print("=" * 60)
    print(r.final)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/l


ENGLISH — AI in Healthcare
The next major breakthrough in medicine won't come from a drug lab.

It’s happening right now in code.

AI in healthcare is no longer a futuristic concept—it’s actively transforming patient outcomes today. 

We are already seeing algorithms:
• Analyze complex medical scans in seconds
• Detect early-stage conditions faster than ever
• Predict patient risks before symptoms fully show up

Crucially, this isn't about replacing doctors. 

By taking on heavy administrative tasks and accelerating diagnostics, AI is giving clinicians back their most valuable asset: time to focus on human care.

Where do you see AI making the biggest impact in healthcare over the next five years? 

#HealthTech #AIinHealthcare #DigitalHealth #FutureOfMedicine

BENGALI — Remote Work Productivity
ঘরে বসে কাজ মানেই সারাদিন ল্যাপটপের স্ক্রিনে আটকে থাকা নয়। 

সঠিক কৌশল জানা থাকলে রিমোট ওয়ার্কেও দ্বিগুণ প্রোডাক্টিভিটি অর্জন করা সম্ভব।

কাজের গতি ও মনোযোগ বাড়াতে ৩টি সহজ অভ্যাস গড়ে তুলুন:

• 

##**7. Save the generated posts :**

- This step saves all the generated LinkedIn posts into a file called `generated_posts.md`. I can use this file to keep the generated posts and also use them later for my LinkedIn posts.

In [9]:
from datetime import datetime

def save_posts_to_markdown(posts, filename="generated_posts.md"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write("# Generated LinkedIn Posts\n")
        f.write(f"*Generated on {datetime.now().strftime('%Y-%m-%d %H:%M')}*\n")
        for p in posts:
            f.write(f"\n---\n### {p.topic} ({p.language})\n\n{p.final}\n")
    print(f"Saved {len(posts)} posts to {filename}")

save_posts_to_markdown(results)

Saved 3 posts to generated_posts.md


##**8. Conclusion :**

- In this project, I built an AI agent using LangChain and Google Gemini that generates professional LinkedIn posts from just a topic and a target language. Rather than a single prompt-response call, I designed the agent around a two-stage reflection pattern — draft, then self-critique and refine — which gives it genuine multi-step reasoning instead of a one-shot
generation.

**Possible extensions I would explore next:**

 - A tool the agent can call to pull real trending hashtags before writing.
 - A tone selector (formal / casual / thought-leader).
 - Wrapping this in a small Streamlit app for non-technical users.

**Tech stack:**
- Python, LangChain (LCEL), Google Gemini API, Google Colab.